# Model hubs

In this expirement we are going to simulate what the model hubs face when they have to deal with a large number of models and users.

In [1]:
from config import MODEL_SUITE, ZIPNN_METHOD
from neuralzip.compressor import Compressor

JS_THRESHOLD = 0.05

 Let's define the compressors:

In [ ]:
# ZipNN baseline
zipnn_local = Compressor(
    "zipnn-local", 
    verify=False,
    threads=12
)

# Official ZipNN implementation
zipnn_src = Compressor(
    "zipnn-src",
    zipnn_method=ZIPNN_METHOD,
    verify=False,
)

# NeuralZip
neuralzip = Compressor(
    "neuralzip",
    verify=False,
    threads=12,
    cluster_js_threshold=JS_THRESHOLD,
)

## Model Storing

The first task we want to test is the storing of models. We want to see how well the compressors can compress a large number of models and how much time it takes to compress them.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import time

import pandas as pd
from IPython.display import display

from neuralzip import HFModel

MODEL_DIR = Path("models")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path("outputs/model_hubs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_storing_compressors = [
    ("ZipNN local", zipnn_local),
    ("ZipNN src", zipnn_src),
    ("NeuralZip", neuralzip),
]

def _model_source(model_config):
    local_subdir = model_config.get("local_subdir")
    if local_subdir:
        local_path = MODEL_DIR / local_subdir
        if local_path.exists():
            return str(local_path), {"local_files_only": True}
    load_kwargs = dict(model_config.get("load_kwargs", {}))
    load_kwargs.setdefault("local_files_only", False)
    return model_config["model_id"], load_kwargs

def _result_row(model_config, compressor_name, result, *, status="ok", error=None):
    raw_bytes = result.get("raw_bytes") if result else None
    compressed_bytes = result.get("compressed_bytes") if result else None
    compression_ratio = result.get("compression_ratio") if result else None
    return {
        "model": model_config["name"],
        "model_id": model_config["model_id"],
        "parameters_b": model_config.get("parameters_b"),
        "dtype": model_config.get("default_dtype"),
        "compressor": compressor_name,
        "method": result.get("method") if result else None,
        "scope": result.get("scope", "full checkpoint") if result else None,
        "status": status,
        "raw_bytes": raw_bytes,
        "compressed_bytes": compressed_bytes,
        "compression_ratio": compression_ratio,
        "compressed_pct": (100 * compression_ratio) if compression_ratio is not None else None,
        "precompute_seconds": result.get("precompute_seconds") if result else None,
        "codebook_setup_seconds": result.get("codebook_setup_seconds") if result else None,
        "encode_seconds": result.get("encode_seconds") if result else None,
        "compression_seconds": result.get("compression_seconds") if result else None,
        "total_compression_seconds": result.get("total_compression_seconds") if result else None,
        "cold_encode_seconds": result.get("cold_encode_seconds") if result else None,
        "decode_seconds": result.get("decode_seconds") if result else None,
        "clusters": result.get("clusters") if result else None,
        "codebooks": len(result.get("codebooks", ())) if result and result.get("codebooks") is not None else None,
        "huffman_chunks": result.get("huffman_chunks") if result else None,
        "raw_chunks": result.get("raw_chunks") if result else None,
        "roundtrip_ok": result.get("roundtrip_ok") if result else None,
        "error": error,
    }

model_storing_results = []
for model_config in MODEL_SUITE:
    source, load_kwargs = _model_source(model_config)
    try:
        load_start = time.perf_counter()
        hf_model = HFModel(source, dtype=model_config["default_dtype"], **load_kwargs)
        load_seconds = time.perf_counter() - load_start
    except Exception as exc:
        for compressor_name, _ in model_storing_compressors:
            row = _result_row(
                model_config,
                compressor_name,
                None,
                status="load-failed",
                error=f"{type(exc).__name__}: {exc}",
            )
            row["load_seconds"] = None
            model_storing_results.append(row)
        pd.DataFrame(model_storing_results).to_csv(OUTPUT_DIR / "model_storing_results.csv", index=False)
        continue

    for compressor_name, compressor in model_storing_compressors:
        try:
            result = compressor.compress(hf_model)
            row = _result_row(model_config, compressor_name, result)
            row["load_seconds"] = load_seconds
        except Exception as exc:
            row = _result_row(
                model_config,
                compressor_name,
                None,
                status="compress-failed",
                error=f"{type(exc).__name__}: {exc}",
            )
            row["load_seconds"] = load_seconds
        model_storing_results.append(row)
        pd.DataFrame(model_storing_results).to_csv(OUTPUT_DIR / "model_storing_results.csv", index=False)

    del hf_model
    gc.collect()

model_storing_df = pd.DataFrame(model_storing_results)
model_storing_df.to_json(OUTPUT_DIR / "model_storing_results.json", orient="records", indent=2)
display(model_storing_df)


## Download

We simulate a download pipeline, this is:

$$
\text{Request} \rightarrow \text{Download} \rightarrow \text{Decompress}
$$

For this our setup is a server that has the compressed model,

## Upload

We simulate a download pipeline, this is:

$$
\text{Request} \rightarrow \text{Compress} \rightarrow \text{Upload}
$$

For this our setup is a server that has the compressed model,